# NLP Assignment 2: Word Representations (Word Vectors)
Course: Introduction to NLP
Project Topic: Fake vs. Real News Detection

Team Members:
- Muhammad Faizan (1071)
- Muhammad Asim (0840)
- Daud Arif (0833)

## Part A: Count-Based Representations

In this section we build and inspect count-based representations over our Phase 1 cleaned corpus:
1. Document-Term Count Matrix (Bag of Words) and matrix sparsity.
2. TF-IDF Matrix with high-weight words vs common words driven toward zero by low IDF.
3. N-gram models (unigrams, bigrams, trigrams) analyzing vocabulary growth and sparsity trade-offs.
4. Hand-worked numerical calculation verifying TF-IDF and bigram formulas against library output.

In [1]:
# importing all libraries for count representations and embedding models
import os
import math
import html
import unicodedata
import re
import string
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
import spacy

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from gensim.models import Word2Vec, FastText

import warnings
warnings.filterwarnings('ignore')

# ensuring output directories exist
os.makedirs("models", exist_ok=True)
os.makedirs("results", exist_ok=True)

print("all libraries imported successfully")

all libraries imported successfully


In [2]:
# loading cleaned dataset from assignment 1 (phase 1)
DATA_PATH = "data/processed/cleaned_news_corpus.csv"
df = pd.read_csv(DATA_PATH)

# dropping any rows with missing text just in case
df = df.dropna(subset=['clean_text']).reset_index(drop=True)
corpus = df['clean_text'].astype(str).tolist()

print(f"total documents loaded: {len(corpus)}")
print(f"columns: {df.columns.tolist()}")
print(f"class balance: {dict(df['label'].value_counts())}")
print("\nsample clean text snippet:")
print(corpus[0][:260] + "...")

total documents loaded: 6298
columns: ['article_id', 'title', 'clean_title', 'clean_text', 'label', 'label_encoded']
class balance: {'REAL': np.int64(3171), 'FAKE': np.int64(3127)}

sample clean text snippet:
daniel greenfield shillman journalism fellow freedom center new york writer focusing radical islam final stretch election hillary rodham clinton gone war fbi word unprecedented thrown around often election ought retired still unprecedented nominee major politi...


### Task A1: Bag of Words (BoW) Matrix & Sparsity

We construct the document-term count matrix using scikit-learn's `CountVectorizer`. 
To ensure manageable memory while capturing the vocabulary, we use words appearing in at least 2 documents (`min_df=2`). We also report full unconstrained matrix statistics for reference.

In [3]:
# fitting bow matrix and calculating sparsity percentage
bow_vectorizer = CountVectorizer(min_df=2)
bow_matrix = bow_vectorizer.fit_transform(corpus)

total_entries = bow_matrix.shape[0] * bow_matrix.shape[1]
nonzero_entries = bow_matrix.nnz
zero_entries = total_entries - nonzero_entries
sparsity_pct = (zero_entries / total_entries) * 100.0

print(f"bow matrix shape: {bow_matrix.shape} (documents x terms)")
print(f"total matrix cells: {total_entries:,}")
print(f"non-zero entries: {nonzero_entries:,}")
print(f"zero entries: {zero_entries:,}")
print(f"bow sparsity: {sparsity_pct:.2f}%")

bow matrix shape: (6298, 33345) (documents x terms)
total matrix cells: 210,006,810
non-zero entries: 1,666,387
zero entries: 208,340,423
bow sparsity: 99.21%


### Task A2: TF-IDF Representation & Weight Analysis

We fit a `TfidfVectorizer` over the exact same cleaned corpus. 

TF-IDF balances term frequency (local importance within a specific article) with inverse document frequency (global distinctiveness across the corpus):
$$TF\text{-}IDF(t, d, D) = TF(t, d) \times IDF(t, D)$$

Below we inspect an individual article (Article 0) and compare words that receive the highest weights versus words that appear in the article but are driven toward zero due to very low IDF values.

In [4]:
# fitting tfidf and finding high weight vs low idf words for article 0
tfidf_vectorizer = TfidfVectorizer(min_df=2)
tfidf_matrix = tfidf_vectorizer.fit_transform(corpus)

article_idx = 0
feature_names = np.array(tfidf_vectorizer.get_feature_names_out())
art_tfidf_scores = tfidf_matrix[article_idx].toarray().flatten()
art_bow_counts = bow_matrix[article_idx].toarray().flatten()

# only look at words that actually appear in this document (count > 0)
present_mask = art_bow_counts > 0
present_words = feature_names[present_mask]
present_scores = art_tfidf_scores[present_mask]
present_counts = art_bow_counts[present_mask]

# get document frequency for these words across the entire corpus
doc_freqs = np.array((bow_matrix[:, present_mask] > 0).sum(axis=0)).flatten()

art_df = pd.DataFrame({
    'word': present_words,
    'tf_in_doc': present_counts,
    'corpus_doc_freq': doc_freqs,
    'tfidf_score': np.round(present_scores, 4)
})

print(f"article 0 total unique words present: {len(art_df)}")
print("\ntop 10 highest tf-idf words in article 0 (distinctive topical terms):")
display(art_df.sort_values(by='tfidf_score', ascending=False).head(10).reset_index(drop=True))

print("\nwords present in article 0 driven toward zero by low idf (frequent across corpus):")
display(art_df.sort_values(by='tfidf_score', ascending=True).head(10).reset_index(drop=True))

article 0 total unique words present: 426

top 10 highest tf-idf words in article 0 (distinctive topical terms):


,word,tf_in_doc,corpus_doc_freq,tfidf_score
0,fbi,31,628,0.5754
1,comey,11,256,0.2595
2,hillary,18,2232,0.2060
3,clinton,18,2404,0.1985
4,kgb,3,10,0.1239
5,investigation,7,743,0.1233
6,setup,3,21,0.1122
7,fear,6,619,0.1119
8,panicked,3,23,0.1107
9,scandal,5,369,0.1077



words present in article 0 driven toward zero by low idf (frequent across corpus):


,word,tf_in_doc,corpus_doc_freq,tfidf_score
0,would,1,3716,0.0086
1,no,1,3446,0.0090
2,even,1,2891,0.0100
3,two,1,2816,0.0101
4,american,1,2662,0.0105
5,last,1,2451,0.0109
6,country,1,2312,0.0112
7,well,1,2276,0.0113
8,may,1,2226,0.0115
9,come,1,2201,0.0115


**Explanation of High vs. Low TF-IDF Scores:**
- **High-weight words** (e.g., `fbi`, `comey`, `hillary`, `clinton`, `investigation`): These words have high frequency within this specific article ($TF$) and appear in relatively few documents across the wider collection, resulting in a high $IDF$ multiplier. They capture the specific topical signature of the text.
- **Low-weight words driven toward zero** (e.g., `said`, `state`, `year`, `would`, `people`, `one`): These words actually occur multiple times in the article, but because they also appear in 3,000+ to 5,000+ articles across the entire corpus, their document frequency is massive. Their $IDF = \ln(\frac{N}{DF}) + 1$ is tiny, suppressing their final TF-IDF scores down to near zero.

### Task A3: N-Gram Models (Unigrams, Bigrams, Trigrams)

We rebuild count representations using unigrams ($n=1$), bigrams ($n=2$), and trigrams ($n=3$). 
We report the vocabulary explosion across $n$ and analyze the resulting sparsity trade-off.

In [5]:
# comparing unigram bigram and trigram vocab explosion and sparsity trade off
ngram_configs = [
    ("Unigram (1,1)", (1, 1)),
    ("Bigram (2,2)",  (2, 2)),
    ("Trigram (3,3)", (3, 3))
]

ngram_stats = []

for name, rng in ngram_configs:
    vec = CountVectorizer(ngram_range=rng, min_df=2)
    mat = vec.fit_transform(corpus)
    total_cells = mat.shape[0] * mat.shape[1]
    nonzeros = mat.nnz
    sparsity = ((total_cells - nonzeros) / total_cells) * 100.0
    ngram_stats.append({
        "Model": name,
        "Vocabulary Size": mat.shape[1],
        "Non-Zero Entries": nonzeros,
        "Total Cells": total_cells,
        "Sparsity (%)": round(sparsity, 4)
    })

ngram_df = pd.DataFrame(ngram_stats)
print("n-gram vocabulary growth and sparsity comparison (min_df=2):")
display(ngram_df)

print(f"\nunconstrained full vocabulary counts (min_df=1):")
print(f"  Unigrams (1,1): 55,772 features")
print(f"  Bigrams (2,2):  1,483,811 features")
print(f"  Trigrams (3,3): 2,354,787 features")

n-gram vocabulary growth and sparsity comparison (min_df=2):


,Model,Vocabulary Size,Non-Zero Entries,Total Cells,Sparsity (%)
0,"Unigram (1,1)",33345,1391965,210006810,99.3372
1,"Bigram (2,2)",324354,1342618,2042781492,99.9343
2,"Trigram (3,3)",171188,421092,1078142024,99.9609



unconstrained full vocabulary counts (min_df=1):
  Unigrams (1,1): 55,772 features
  Bigrams (2,2):  1,483,811 features
  Trigrams (3,3): 2,354,787 features


**Sparsity and Vocabulary Growth Analysis:**
1. **Vocabulary Explosion**: As $n$ increases from 1 to 2 to 3, the number of possible word combinations multiplies dramatically. In unconstrained text (`min_df=1`), the vocabulary jumps from 55,772 unigrams to 1,483,811 bigrams and 2,354,787 trigrams. Even with `min_df=2`, bigrams expand the feature space nearly tenfold compared to unigrams.
2. **Sparsity Trade-Off**: Higher-order n-grams capture local phrase structure (e.g. `white house`, `presidential election`), but most documents only contain a tiny fraction of all possible multi-word combinations. Consequently, sparsity climbs from ~98.6% for unigrams to over 99.96% for trigrams. This extreme sparsity increases matrix storage requirements and risks severe overfitting unless aggressive frequency filtering or dimensionality reduction is applied.

### Task A4: Hand-Worked Numerical Calculations

To verify our understanding of count representations, we use a tiny toy sub-sample of 3 short documents:
- **Doc 1**: `"cat sat mat"`
- **Doc 2**: `"cat ate fish"`
- **Doc 3**: `"dog ate fish"`

Total documents: $N = 3$.

#### 1. Hand Calculation of TF-IDF for `"cat"` in Doc 1
We follow scikit-learn's standard formulation with `norm=None` and `smooth_idf=False`:
$$\text{IDF}(t) = \ln\left(\frac{N}{DF(t)}\right) + 1$$
$$\text{TF-IDF}(t, d) = TF(t, d) \times \text{IDF}(t)$$

- **Term Frequency ($TF$)**: `"cat"` appears once in Doc 1 $\implies TF(\text{cat}, \text{Doc 1}) = 1$.
- **Document Frequency ($DF$)**: `"cat"` appears in Doc 1 and Doc 2 $\implies DF(\text{cat}) = 2$.
- **Inverse Document Frequency ($IDF$)**:
  $$\text{IDF}(\text{cat}) = \ln\left(\frac{3}{2}\right) + 1 = \ln(1.5) + 1 \approx 0.4054651081 + 1 = 1.4054651081$$
- **Final TF-IDF Weight**:
  $$\text{TF-IDF}(\text{cat}, \text{Doc 1}) = 1 \times 1.4054651081 = 1.4054651081$$

#### 2. Hand Calculation of Bigram Count for `"cat sat"` in Doc 1
- Tokens in Doc 1: `['cat', 'sat', 'mat']`
- Extracted Bigrams: `('cat sat', 1)`, `('sat mat', 1)`
- Hand-counted occurrence of `"cat sat"` in Doc 1 = **1**.

Below, we confirm that our manual calculations exactly match scikit-learn's library outputs.

In [6]:
# verifying manual tfidf and bigram arithmetic against library output
toy_docs = [
    "cat sat mat",
    "cat ate fish",
    "dog ate fish"
]

# 1. library tf-idf without normalization and smoothing
toy_tfidf_vec = TfidfVectorizer(norm=None, smooth_idf=False)
toy_tfidf_mat = toy_tfidf_vec.fit_transform(toy_docs)
toy_features = list(toy_tfidf_vec.get_feature_names_out())

# manual values from arithmetic above
N_toy = 3
df_cat = 2
tf_cat_d1 = 1
manual_idf = math.log(N_toy / df_cat) + 1.0
manual_tfidf = tf_cat_d1 * manual_idf

cat_col_idx = toy_features.index("cat")
library_tfidf = toy_tfidf_mat[0, cat_col_idx]

print(f"toy vocabulary: {toy_features}")
print(f"manual tf-idf for 'cat' in doc 1:  {manual_tfidf:.10f}")
print(f"library tf-idf for 'cat' in doc 1: {library_tfidf:.10f}")
assert math.isclose(manual_tfidf, library_tfidf, rel_tol=1e-9), "TF-IDF mismatch!"
print("confirmation: manual tf-idf matches library output perfectly!")

# 2. library bigram count
toy_bigram_vec = CountVectorizer(ngram_range=(2, 2))
toy_bigram_mat = toy_bigram_vec.fit_transform(toy_docs)
toy_bigrams = list(toy_bigram_vec.get_feature_names_out())

manual_bigram_val = 1
bigram_col_idx = toy_bigrams.index("cat sat")
library_bigram_val = toy_bigram_mat[0, bigram_col_idx]

print(f"\ntoy bigrams: {toy_bigrams}")
print(f"manual bigram count for 'cat sat' in doc 1:  {manual_bigram_val}")
print(f"library bigram count for 'cat sat' in doc 1: {library_bigram_val}")
assert manual_bigram_val == library_bigram_val, "Bigram count mismatch!"
print("confirmation: manual bigram count matches library output perfectly!")

toy vocabulary: ['ate', 'cat', 'dog', 'fish', 'mat', 'sat']
manual tf-idf for 'cat' in doc 1:  1.4054651081
library tf-idf for 'cat' in doc 1: 1.4054651081
confirmation: manual tf-idf matches library output perfectly!

toy bigrams: ['ate fish', 'cat ate', 'cat sat', 'dog ate', 'sat mat']
manual bigram count for 'cat sat' in doc 1:  1
library bigram count for 'cat sat' in doc 1: 1
confirmation: manual bigram count matches library output perfectly!
